# Polarimetric orientation compensation: Python–C parity on ALOS-1

This notebook runs C-PolSARpro and Python polarimetric orientation compensation on the San Francisco ALOS-1 scattering matrix, then compares the estimated angle and corrected S matrix.

In [ ]:
%load_ext autoreload
%autoreload 2

import shutil
import subprocess
from pathlib import Path

import dask
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.dev.io import read_psp_bin
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import orientation_compensation

c_bin = Path("/home/c_psp/Soft/bin/data_process_sngl")
estimation_executable = c_bin / "orientation_estimation.exe"
correction_executable = c_bin / "orientation_correction.exe"
input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1")
output_dir = Path("/data/psp/res/orientation_compensation_alos1_c")
output_dir.mkdir(parents=True, exist_ok=True)

boxcar_size = (3, 3)

## Load the common scattering-matrix input

Python reads the NetCDF representation while C reads the equivalent PolSARpro S2 bands.

In [ ]:
S = open_netcdf_beam(input_file)
nrows, ncols = S.sizes["y"], S.sizes["x"]
mask_file = input_dir / "mask_valid_pixels.bin"
valid = read_psp_bin(mask_file).astype(bool)
valid_da = xr.DataArray(valid, dims=("y", "x"), coords=S.coords)
S = S.where(valid_da)
S

## Run C-PolSARpro estimation and correction

The ALOS-1 NetCDF conversion maps Python `hv` to legacy `s21.bin` and Python `vh` to `s12.bin`.

In [ ]:
angle_file = output_dir / "orientation_estimation.bin"
memory_error_file = output_dir / "memory_error.txt"

estimation_command = [
    str(estimation_executable),
    "-id", str(input_dir),
    "-od", str(output_dir),
    "-iodf", "S2",
    "-nwr", str(boxcar_size[0]),
    "-nwc", str(boxcar_size[1]),
    "-ofr", "0",
    "-ofc", "0",
    "-fnr", str(nrows),
    "-fnc", str(ncols),
    "-errf", str(memory_error_file),
    "-mask", str(mask_file),
]
subprocess.run(estimation_command, check=False)
assert angle_file.is_file()

correction_command = [
    str(correction_executable),
    "-id", str(input_dir),
    "-od", str(output_dir),
    "-if", str(angle_file),
    "-iodf", "S2",
    "-ofr", "0",
    "-ofc", "0",
    "-fnr", str(nrows),
    "-fnc", str(ncols),
    "-errf", str(memory_error_file),
    "-mask", str(mask_file),
]
subprocess.run(correction_command, check=False)

c_files = {
    name: output_dir / f"s{index}.bin"
    for name, index in {"hh": "11", "hv": "21", "vh": "12", "vv": "22"}.items()
}
assert all(path.is_file() for path in c_files.values())
shutil.copy(input_dir / "config.txt", output_dir / "config.txt")

## Run the Python implementation

In [ ]:
corrected_lazy, orientation_lazy = orientation_compensation(
    S, boxcar_size=boxcar_size
)
with ProgressBar():
    corrected_py, orientation_py = dask.compute(
        corrected_lazy, orientation_lazy
    )

## Read the C outputs

In [ ]:
orientation_c = np.where(valid, read_psp_bin(angle_file), np.nan)
corrected_c = {
    name: np.where(
        valid,
        read_psp_bin(path, dtype="complex64"),
        np.nan + 1j * np.nan,
    )
    for name, path in c_files.items()
}

assert orientation_c.shape == orientation_py.orientation_angle.shape
for name in corrected_py.data_vars:
    assert corrected_c[name].shape == corrected_py[name].shape

## Numerical comparison

In [ ]:
summarize_metrics(
    orientation_py,
    {"orientation_angle": orientation_c},
    verbose=False,
)

In [ ]:
summarize_metrics(corrected_py, corrected_c, verbose=False)

## Inspect spatial error patterns

In [ ]:
visualize_errors(
    orientation_py,
    {"orientation_angle": orientation_c},
    sub_az=16,
)
visualize_errors(corrected_py, corrected_c, sub_az=16)